# 04 — Avaliação final no teste, ensemble e impacto financeiro (Fase 5)

- **Cenários B** (principal, realista) **e C** (sem o atalho do esvaziamento).
- Os 5 modelos são retreinados com `FRAC_FINAL` das legítimas (1.0 = todas) e avaliados na **validação e no teste**.
- **Ensemble** (pesos e stacking ajustados só na validação, em 2 blocos temporais; o teste nunca entra no ajuste):
  stacking com todos os 5 modelos, stacking só com supervisionados (isola a contribuição dos modelos de anomalia) e média ponderada.
- **Impacto financeiro**: corte escolhido na validação, aplicado no teste, para vários custos de investigação.

Runtime: **CPU** serve (a GPU só acelera o XGBoost e o Autoencoder). Pode levar de 15 a 40 min com `FRAC_FINAL = 1.0`.
Para um teste rápido, use `FRAC_FINAL = 0.2` e `SALVAR_RF = False`.

In [ ]:
import os, sys, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fraud-detection-paysim'
except ImportError:
    PROJECT_DIR = os.getcwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

import numpy as np, pandas as pd, joblib
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

from fraud.config import load_config
cfg = load_config()
y = cfg['data']['target']
FIG = os.path.join(cfg['paths']['reports_dir'], 'figures'); os.makedirs(FIG, exist_ok=True)
MODELS = cfg['paths']['models_dir']; os.makedirs(MODELS, exist_ok=True)
SCORES = os.path.join(cfg['paths']['reports_dir'], 'scores'); os.makedirs(SCORES, exist_ok=True)
pd.options.display.float_format = '{:,.4f}'.format
print('Projeto:', PROJECT_DIR)

In [ ]:
FRAC_FINAL = 1.0        # fração das legítimas no treino final
SALVAR_RF = False       # o Random Forest salvo fica muito grande
CENARIOS_FINAIS = {'B': 'sem_saldos_pos', 'C': 'sem_esvaziamento'}
CUSTOS = [50, 500, 5000, 50000]   # custo de investigação por alerta (grade de sensibilidade)
CATCH = cfg['costs']['investigation_catch_rate']
FINAL = os.path.join(MODELS, 'final'); os.makedirs(FINAL, exist_ok=True)

## 1. Dados e split

In [ ]:
from fraud.data import load_parquet, dev_sample
from fraud.split import make_split
from fraud.anomaly import in_scope, AnomalyPrep, fill_out_of_scope, build_autoencoder, reconstruction_error
from fraud.features import build_features
from fraud.metrics import evaluate

df = load_parquet(cfg['paths']['parquet'])
train, val, test = make_split(df, cfg)
train_s = dev_sample(train, FRAC_FINAL, target=y, seed=cfg['seed']) if FRAC_FINAL < 1 else train
n_pos = int(train_s[y].sum()); n_neg = len(train_s) - n_pos
tr_legit = train_s[in_scope(train_s) & (train_s[y] == 0)]
m_val, m_test = in_scope(val), in_scope(test)
for nome, p in [('treino usado', train_s), ('validação', val), ('teste', test)]:
    print(f"{nome}: {len(p):,} linhas | {int(p[y].sum()):,} fraudes ({100*p[y].mean():.4f}%)")

## 2. Treino final dos 5 modelos e scores na validação e no teste

In [ ]:
import subprocess, keras
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
try:
    HAS_GPU = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    HAS_GPU = False
print('GPU:', HAS_GPU)

def sup_models():
    rf_rows = min(0.5, 383000 / max(len(train_s), 1))   # ~383 mil linhas por árvore, como no desenvolvimento
    return {
        'logreg': make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=500)),
        'random_forest': RandomForestClassifier(n_estimators=150, max_depth=14, min_samples_leaf=5,
                                                max_samples=rf_rows, class_weight='balanced_subsample',
                                                n_jobs=-1, random_state=cfg['seed']),
        'xgboost': XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.1, subsample=0.8,
                                 colsample_bytree=0.8, scale_pos_weight=n_neg / max(n_pos, 1),
                                 tree_method='hist', device='cuda' if HAS_GPU else 'cpu',
                                 eval_metric='aucpr', random_state=cfg['seed']),
    }

S = {'val': {}, 'test': {}}      # S[split][(modelo, cenario)] = score
tempos = []
for letra, variante in CENARIOS_FINAIS.items():
    Xtr, ytr = build_features(train_s, variante), train_s[y]
    Xva, Xte = build_features(val, variante), build_features(test, variante)
    for nome, m in sup_models().items():
        t0 = time.time(); m.fit(Xtr, ytr)
        S['val'][(nome, letra)] = m.predict_proba(Xva)[:, 1]
        S['test'][(nome, letra)] = m.predict_proba(Xte)[:, 1]
        tempos.append((letra, nome, round(time.time() - t0, 1)))
        print(f'ok cenário {letra} | {nome:16s} | {tempos[-1][2]:7.1f}s')
        if letra == 'B' and (nome != 'random_forest' or SALVAR_RF):
            joblib.dump(m, os.path.join(FINAL, f'{nome}_B.joblib'), compress=3)
    prep = AnomalyPrep(variante).fit(tr_legit)
    Ptr, Pva, Pte = prep.transform(tr_legit), prep.transform(val[m_val]), prep.transform(test[m_test])
    t0 = time.time()
    iso = IsolationForest(n_estimators=200, max_samples=1024, n_jobs=-1, random_state=cfg['seed']).fit(Ptr)
    S['val'][('isolation_forest', letra)] = fill_out_of_scope(-iso.score_samples(Pva), m_val)
    S['test'][('isolation_forest', letra)] = fill_out_of_scope(-iso.score_samples(Pte), m_test)
    tempos.append((letra, 'isolation_forest', round(time.time() - t0, 1)))
    print(f'ok cenário {letra} | isolation_forest  | {tempos[-1][2]:7.1f}s')
    t0 = time.time()
    ae = build_autoencoder(Ptr.shape[1], seed=cfg['seed'])
    ae.fit(Ptr, Ptr, epochs=15, batch_size=2048, validation_split=0.1, verbose=0,
           callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])
    S['val'][('autoencoder', letra)] = fill_out_of_scope(reconstruction_error(ae, Pva), m_val)
    S['test'][('autoencoder', letra)] = fill_out_of_scope(reconstruction_error(ae, Pte), m_test)
    tempos.append((letra, 'autoencoder', round(time.time() - t0, 1)))
    print(f'ok cenário {letra} | autoencoder       | {tempos[-1][2]:7.1f}s')
    if letra == 'B':
        joblib.dump({'prep': prep, 'model': iso}, os.path.join(FINAL, 'isolation_forest_B.joblib'), compress=3)
        joblib.dump(prep, os.path.join(FINAL, 'autoencoder_B_prep.joblib'), compress=3)
        ae.save(os.path.join(FINAL, 'autoencoder_B.keras'))

for sp in S:
    for (nome, letra), s in S[sp].items():
        np.save(os.path.join(SCORES, f'{sp}_{nome}_{letra}_final.npy'), s.astype('float32'))
yv, yt = val[y].values, test[y].values
av, at = val['amount'].values, test['amount'].values

## 3. Resultado dos 5 modelos no teste

In [ ]:
ordem = ['logreg', 'random_forest', 'xgboost', 'isolation_forest', 'autoencoder']
linhas = []
for sp, yy, aa in [('val', yv, av), ('test', yt, at)]:
    for (nome, letra), s in S[sp].items():
        r = evaluate(yy, s, amount=aa); r.update({'split': sp, 'cenario': letra, 'modelo': nome}); linhas.append(r)
res = pd.DataFrame(linhas).set_index(['split', 'cenario', 'modelo'])
res.to_csv(os.path.join(cfg['paths']['reports_dir'], 'final_5_modelos.csv'))
for col, nome in [('pr_auc', 'PR-AUC'), ('bestF1_recall', 'Recall (melhor F1)'),
                  ('bestF1_value_recall', 'Recall em VALOR (melhor F1)'), ('bestF1_fpr', 'FPR (melhor F1)')]:
    print(nome)
    display(res[col].unstack('modelo')[ordem])

## 4. Ensemble
O score de cada modelo é levado à escala `-log10(1 - ECDF_validação)`. Os pesos e o stacking são ajustados na validação
em **2 blocos temporais** (ajusta em um, pontua no outro), o que dá uma avaliação justa na própria validação.
Depois o ajuste final usa a validação inteira e o resultado é medido no teste.

In [ ]:
from fraud.ensemble import (fit_tail_ranks, stack_matrix, first_half_mask, stack_oof,
                            stack_fit_predict, weights_oof, best_weights)
fold = first_half_mask(val['step'].values)
E = {}   # E[(nome_ensemble, cenario)] = (score_val, score_test)
pesos = {}
for letra in CENARIOS_FINAIS:
    sv = {n: S['val'][(n, letra)] for n in ordem}
    st = {n: S['test'][(n, letra)] for n in ordem}
    ranks = fit_tail_ranks(sv)
    for nome_e, nomes in [('stack_5', ordem), ('stack_supervisionados', ordem[:3])]:
        Qv, Qt = stack_matrix(sv, ranks, nomes), stack_matrix(st, ranks, nomes)
        E[(nome_e, letra)] = (stack_oof(Qv, yv, fold), stack_fit_predict(Qv, yv, Qt))
    Qv, Qt = stack_matrix(sv, ranks, ordem), stack_matrix(st, ranks, ordem)
    w = best_weights(Qv, yv)
    pesos[letra] = dict(zip(ordem, w))
    E[('media_ponderada_5', letra)] = (weights_oof(Qv, yv, fold), Qt @ w)
print('Pesos da média ponderada (ajuste final na validação):')
display(pd.DataFrame(pesos))

from sklearn.metrics import average_precision_score as AP
linhas = []
for (nome_e, letra), (ev, et) in E.items():
    r = evaluate(yt, et, amount=at)
    linhas.append({'cenario': letra, 'modelo': nome_e, 'pr_auc_val_fora_da_amostra': AP(yv, ev),
                   'pr_auc_teste': r['pr_auc'], 'recall_valor_teste': r['bestF1_value_recall'],
                   'fpr_teste': r['bestF1_fpr']})
for letra in CENARIOS_FINAIS:
    for n in ordem:
        r = res.loc[('test', letra, n)]
        linhas.append({'cenario': letra, 'modelo': n, 'pr_auc_val_fora_da_amostra': res.loc[('val', letra, n), 'pr_auc'],
                       'pr_auc_teste': r['pr_auc'], 'recall_valor_teste': r['bestF1_value_recall'],
                       'fpr_teste': r['bestF1_fpr']})
ens = pd.DataFrame(linhas).set_index(['cenario', 'modelo']).sort_index()
ens.to_csv(os.path.join(cfg['paths']['reports_dir'], 'final_ensemble.csv'))
display(ens)

## 5. Impacto financeiro (regra de dois níveis: sinalizar se score ≥ corte)
O corte maximiza o benefício líquido na **validação** e é aplicado no **teste**. Falsos positivos são reponderados para a
taxa de fraude real do PaySim (0,129%), porque a taxa do teste é bem maior. Valores em unidades monetárias do PaySim.
A versão de três níveis (com autenticação adicional) vem na Fase 6.

In [ ]:
from fraud.costs import best_threshold, financial_at_threshold, legit_weight_for
wv, wt = legit_weight_for(yv), legit_weight_for(yt)
print(f'pesos de reponderação: validação {wv:.2f} | teste {wt:.2f}')

candidatos = {(n, l): (S['val'][(n, l)], S['test'][(n, l)]) for l in CENARIOS_FINAIS for n in ordem}
candidatos.update({(n, l): v for (n, l), v in E.items()})
linhas = []
for (nome, letra), (sv_, st_) in candidatos.items():
    for custo in CUSTOS:
        thr = best_threshold(yv, av, sv_, custo, CATCH, wv)
        r = financial_at_threshold(yt, at, st_, thr, custo, CATCH, wt)
        r.update({'cenario': letra, 'modelo': nome, 'custo_por_alerta': custo})
        linhas.append(r)
fin = pd.DataFrame(linhas).set_index(['cenario', 'custo_por_alerta', 'modelo']).sort_index()
fin.to_csv(os.path.join(cfg['paths']['reports_dir'], 'final_impacto_financeiro.csv'))
cols = ['alertas', 'fraudes_pegas', 'falsos_positivos_reponderados', 'valor_poupado_%', 'custo_investigacao', 'beneficio_liquido']
pd.options.display.float_format = '{:,.2f}'.format
for letra in CENARIOS_FINAIS:
    for custo in CUSTOS:
        print(f'Cenário {letra} | custo de investigação = {custo:,}')
        display(fin.loc[(letra, custo), cols].sort_values('beneficio_liquido', ascending=False).head(6))
print('Total de fraude no teste:', f"{at[yt == 1].sum():,.0f}", '| fraudes:', int(yt.sum()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, letra in zip(axes, CENARIOS_FINAIS):
    sel = fin.loc[letra].reset_index()
    for nome in ['xgboost', 'random_forest', 'logreg', 'stack_5', 'stack_supervisionados']:
        d = sel[sel['modelo'] == nome]
        ax.plot(d['custo_por_alerta'], d['valor_poupado_%'], marker='o', label=nome)
    ax.set_xscale('log'); ax.set_title(f'Cenário {letra}: % do valor fraudado poupado (teste)')
    ax.set_xlabel('custo de investigação por alerta'); ax.legend(fontsize=8)
axes[0].set_ylabel('% poupado')
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'final_valor_poupado.png'), dpi=130); plt.show()

## 6. Exportar todas as saídas deste notebook em um .txt

In [ ]:
import json
from google.colab import _message, files
nb = _message.blocking_request('get_ipynb', request='', timeout_sec=120)['ipynb']
partes = []
for c in nb['cells']:
    if c['cell_type'] != 'code':
        continue
    for o in c.get('outputs', []):
        if o.get('output_type') == 'stream':
            partes.append(''.join(o['text']))
        elif 'data' in o and 'text/plain' in o['data']:
            partes.append(''.join(o['data']['text/plain']))
txt = os.path.join(cfg['paths']['reports_dir'], 'saidas_notebook_04.txt')
open(txt, 'w', encoding='utf-8').write('\n'.join(partes))
files.download(txt)

## O que enviar de volta
O arquivo `saidas_notebook_04.txt` baixado na seção 6 e a imagem `final_valor_poupado.png` (em `reports/figures`).